In [ ]:
import numpy as np
import cupy as cp
from tanalysis import improcess, stitching, stitching_gpu, traj_analysis
import tifffile as tiff
import os
import matplotlib.pyplot as plt

In [ ]:
dirn = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Originals\24h_CXCL10_Conc10_z5_t8h-0.tif"
imgs, names, info = improcess.imread(dirn, tiles=False, gpu=False)
img = imgs[0][22][4]
m,n = img.shape
U, S, V = np.linalg.svd(img)
k = int(0.1*m)
sigma = np.eye(k)*S[:k]
nU = U[:,:k]
nV = V[:k,:]
print(nU.shape, sigma.shape, nV.shape)
nimg = nU@sigma@nV
savedir = r"C:\Users\pcanaleta\Desktop"
plt.imsave(os.path.join(savedir, 'original.png'), img)
plt.imsave(os.path.join(savedir, 'svd.png'), nimg)
#print(U,S,V)

In [ ]:
dirn = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Originals"
for fname in os.listdir(dirn):
    dirname = os.path.join(dirn, fname)
    imgs, names, info = improcess.imread(dirname, tiles=True, gpu=True)
    positions = info['mosaic_position']
    translations_list = stitching_gpu.translationComputation(imgs, positions, n=20, n_frames=10)
    res_img_list = stitching_gpu.image_reconstruction(imgs, positions, translations_list)
    print(res_img_list[0].shape)
    newname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Results"
    im = cp.asnumpy(res_img_list[0])
        
    for name in names:
        tiff.imwrite(
            fr'{newname}\{name}.tif', 
            im[:,:,:,:].astype(np.uint8), 
            imagej=True,
            metadata={
                'axes':'TZYX'
            })


In [ ]:
dname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD1.T\EXP.HD1.15.6\EXP.HD1.15.6.1_Processed\24h\Originals"
model = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\training_process\processed_imgs\models\processed_imgs"

for fname in os.listdir(dname):
    dirname = os.path.join(dname, fname)
    imgs, names, info = improcess.imread(dirname)
    savedir = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD1.T\EXP.HD1.15.6\EXP.HD1.15.6.1_Processed\24h\Results"
    temp_savedir = improcess.cellposeseg(imgs, 3, names, savedir, modelpath=model)